# 05 - LightGBM 滚动因子融合
训练带 purge 间隔的滚动模型，保存最新检查点，并输出每日股票预测分数。完整说明见 `docs/运行手册.md`。

In [ ]:
from pathlib import Path
import os
if not Path('configs/training_config.yaml').exists():
    repo=os.environ.get('ALPHAMINING_REPO_URL','https://github.com/JacksonChiy/AlphaMining-GFlowNet-AlphaEval.git')
    !git clone $repo
    %cd AlphaMining-GFlowNet-AlphaEval

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA 状态:',torch.cuda.is_available(),'CUDA 运行时:',torch.version.cuda)
if torch.cuda.is_available():
    p=torch.cuda.get_device_properties(0); print('GPU 型号:',p.name); print('GPU 显存（GB）:',round(p.total_memory/1024**3,2))
else: print('GPU 不可用；GPU 显存（GB）: 0')

In [ ]:
import yaml,pandas as pd
config=yaml.safe_load(open('configs/training_config.yaml',encoding='utf-8'))
from src.model import LightGBMConfig,LightGBMFusion
evaluation=pd.read_csv('results/alpha_eval_result.csv')
selected=evaluation.loc[evaluation.dpp_selected.astype(bool),'factor'].tolist()
fusion=LightGBMFusion(LightGBMConfig(**config['lightgbm']))
prediction=fusion.fit_predict(pd.read_pickle(config['dataset']['output']),pd.read_pickle('results/alpha_factor_matrix.pkl'),selected)
display(prediction.tail(30))
loaded=LightGBMFusion.load('results/lightgbm/lgbm_model.joblib')
print('重载成功；特征数:',len(loaded['features']))